# Аномалии и риск отказа


In [1]:
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sqlalchemy import create_engine, text

import os
postgres_url = lambda: f"postgresql+psycopg2://{os.getenv('POSTGRES_USER', 'admin')}:{os.getenv('POSTGRES_PASSWORD', 'admin')}@{os.getenv('POSTGRES_HOST', 'postgres')}:{os.getenv('POSTGRES_PORT', '5432')}/{os.getenv('POSTGRES_DB', 'oil_analytics')}"

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "docker-compose.yml").exists())
FEATURES = ["vibration", "temperature", "current", "rpm", "pressure"]


In [2]:
def run_risk_model():

    engine = create_engine(postgres_url())
    sensors = pd.read_sql("SELECT * FROM pump_sensors", engine, parse_dates=["timestamp"])
    failures = pd.read_sql("SELECT * FROM pump_failures", engine, parse_dates=["failure_date"])
    sensors[FEATURES] = sensors.groupby("pump_id")[FEATURES].transform(lambda x: x.ffill().bfill().fillna(x.median()))
    for column in FEATURES:
        sensors[f"{column}_z"] = sensors.groupby("pump_id")[column].transform(lambda x: ((x - x.mean()) / x.std(ddof=0)).fillna(0))
    detector = IsolationForest(contamination=0.03, random_state=42)
    sensors["anomaly_flag"] = (detector.fit_predict(sensors[FEATURES]) == -1).astype(int)
    sensors["anomaly_score"] = -detector.score_samples(sensors[FEATURES])
    sensors["failure_within_24h"] = 0
    for row in failures.itertuples(index=False):
        mask = (
            (sensors["pump_id"] == row.pump_id)
            & (sensors["timestamp"] <= row.failure_date)
            & (sensors["timestamp"] >= row.failure_date - pd.Timedelta(hours=24))
        )
        sensors.loc[mask, "failure_within_24h"] = 1
    risk_features = FEATURES + ["anomaly_score"]
    x_train, x_test, y_train, y_test = train_test_split(
        sensors[risk_features],
        sensors["failure_within_24h"],
        test_size=0.25,
        random_state=42,
        stratify=sensors["failure_within_24h"],
    )
    model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
    model.fit(x_train, y_train)
    prediction = model.predict(x_test)
    metrics = classification_report(y_test, prediction, output_dict=True, zero_division=0)
    metrics["roc_auc"] = float(roc_auc_score(y_test, model.predict_proba(x_test)[:, 1]))
    sensors["failure_risk_score"] = model.predict_proba(sensors[risk_features])[:, 1]
    output = PROJECT_ROOT / "models"
    output.mkdir(exist_ok=True)
    joblib.dump(detector, output / "pump_isolation_forest.joblib")
    joblib.dump(model, output / "pump_failure_classifier.joblib")
    (output / "pump_failure_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    with engine.begin() as connection:
        connection.execute(text("DROP TABLE IF EXISTS mart_pump_risk"))
    sensors.to_sql("mart_pump_risk", engine, if_exists="replace", index=False)
    print(json.dumps({"created": "mart_pump_risk", "roc_auc": metrics["roc_auc"]}, indent=2))



run_risk_model()


{
  "created": "mart_pump_risk",
  "roc_auc": 1.0
}
